# Transformer churn training

使用 `feature_pipeline.py` + `transformer_model.py` 完成特征处理、Transformer 训练、验证和提交文件生成。

In [ ]:
import os, sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

sys.path.append(os.getcwd())

from feature_pipeline import prepare_datasets
from transformer_model import (
    TrainingConfig,
    build_user_sequences,
    make_dataloaders,
    ChurnTransformer,
    train_model,
    predict_proba,
    UserSequenceDataset,
    collate_batch,
)


In [ ]:
BASE_PATH = os.getcwd()
train_path = f'{BASE_PATH}/churn-prediction-25-26/train.parquet'
test_path = f'{BASE_PATH}/churn-prediction-25-26/test.parquet'

config = TrainingConfig(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.3,
    max_seq_len=1000,
    batch_size=32,
    lr=3e-4,
    weight_decay=7e-4,
    epochs=50,
    use_cosine_decay=True,
    eta_min_factor=0.1,
    warmup_epochs=2,
    use_focal_loss=True,
    focal_gamma=2.0,
    early_stop_patience=3,
    early_stop_min_delta=5e-4,
)

train_df, val_df, test_df, labels, artifacts = prepare_datasets(
    train_path,
    test_path,
    val_ratio=0.2,
    random_state=42,
    truncate_buffer_min=5, # prevent leakage
    truncate_buffer_frac=0.1,
    cutoff_time='2018-11-10',  #this is very important, 
)
label_dict = labels.to_dict()
print('train rows', len(train_df), 'val rows', len(val_df), 'test rows', len(test_df))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_sequences = build_user_sequences(train_df, artifacts.numeric_cols, config.max_seq_len)
val_sequences = build_user_sequences(val_df, artifacts.numeric_cols, config.max_seq_len)
train_loader, val_loader = make_dataloaders(train_sequences, val_sequences, label_dict, config)

pos_weight = float((labels == 0).sum() / (labels == 1).sum())
model = ChurnTransformer(
    num_numeric=len(artifacts.numeric_cols),
    num_pages=len(artifacts.page_categories),
    num_metro=len(artifacts.metro_mapping),
    num_state=len(artifacts.state_mapping),
    num_device=len(artifacts.device_mapping),
    config=config,
).to(device)
print(model)


In [ ]:
train_losses, val_losses, model = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=config.epochs,
    lr=config.lr,
    weight_decay=config.weight_decay,
    pos_weight=pos_weight,
    use_cosine_decay=config.use_cosine_decay,
    eta_min_factor=config.eta_min_factor,
    warmup_epochs=config.warmup_epochs,
    use_focal_loss=config.use_focal_loss,
    focal_gamma=config.focal_gamma,
    early_stop_patience=config.early_stop_patience,
    early_stop_min_delta=config.early_stop_min_delta,
)

print('done training')


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()


In [ ]:
val_probs = predict_proba(model, val_loader, device)
val_index = list(val_sequences.keys())
val_true = labels.loc[val_index]
val_pred = pd.Series(val_probs).reindex(val_index)
val_pred_binary = (val_pred > 0.5).astype(int)
acc = accuracy_score(val_true, val_pred_binary)
auc = roc_auc_score(val_true, val_pred)
print(f'Validation accuracy: {acc:.4f}, ROC-AUC: {auc:.4f}')


In [ ]:
test_sequences = build_user_sequences(test_df, artifacts.numeric_cols, config.max_seq_len)
test_dataset = UserSequenceDataset(test_sequences, labels=None, user_ids=list(test_sequences.keys()))
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    collate_fn=collate_batch,
)
test_probs = predict_proba(model, test_loader, device)
submission = pd.DataFrame({
    'id': list(test_probs.keys()),
    'target': (pd.Series(test_probs) > 0.5).astype(int).values
})
submission['id'] = pd.to_numeric(submission['id'], errors='ignore')
submission.to_csv('submission.csv', index=False)
submission.head()
